In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from phantominator import shepp_logan

from DcTNN.tnn import cascadeNet, axVIT, patchVIT
from dc.dc import FFT_DC, KSpace_DC, fft_2d, ifft_2d
from dataset import MRIDataset, load_mask

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
CHECKPOINT_BASE = '../Experiments'
DATA_DIR        = '/scratch/user/uqanag/OASIS/keras_png_slices_train'
MASK_DIR        = '../masks'
N               = 320
ACCEL           = 8

In [ ]:
def build_model_from_cfg(cfg):
    k_space  = cfg.get('k_space_learning', False)
    encoders = cfg.get('encoders', ['patch', 'patch', 'patch'])
    ch       = 2 if k_space else cfg.get('num_channels', 1)

    enc_list, enc_args = [], []
    for name in encoders:
        if name == 'axial':
            enc_list.append(axVIT)
            enc_args.append(dict(
                layerNo=cfg.get('layer_no', 1), numCh=ch, d_model=None,
                nhead=cfg.get('nhead_axial', 8),
                num_encoder_layers=cfg.get('num_encoder_layers', 2),
                dim_feedforward=None, pos_emb_type=cfg.get('pos_emb_type', 'APE'),
                rope_theta=cfg.get('rope_theta', 100.0),
                rope_mixed_rotate=cfg.get('rope_mixed_rotate', True),
            ))
        else:
            enc_list.append(patchVIT)
            enc_args.append(dict(
                patch_size=cfg.get('patch_size', 16),
                kaleidoscope=(name == 'kaleidoscope'),
                layerNo=cfg.get('layer_no', 1), numCh=ch,
                nhead=cfg.get('nhead_patch', 8),
                num_encoder_layers=cfg.get('num_encoder_layers', 2),
                dim_feedforward=None, d_model=None,
                pos_emb_type=cfg.get('pos_emb_type', 'APE'),
                rope_theta=cfg.get('rope_theta', 100.0),
                rope_mixed_rotate=cfg.get('rope_mixed_rotate', True),
            ))

    dc_func  = KSpace_DC if k_space else FFT_DC
    use_lamb = (cfg.get('lambda_schedule', 'none') == 'none')
    return cascadeNet(cfg.get('image_size', 320), enc_list, enc_args,
                      dc_func, use_lamb, k_space_learning=k_space)


def load_experiment(name):
    exp_dir     = os.path.join(CHECKPOINT_BASE, name)
    config_path = os.path.join(exp_dir, 'config.json')
    ckpt_path   = os.path.join(exp_dir, 'best_model.pth')

    with open(config_path) as f:
        cfg = json.load(f)

    model = build_model_from_cfg(cfg).to(device)

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        ep   = ckpt.get('epoch', '?')
        psnr = ckpt.get('val_psnr', float('nan'))
        print(f'  {name:45s}  epoch={ep}, val_psnr={psnr:.2f} dB')
    else:
        print(f'  {name:45s}  WARNING — no checkpoint, using random weights')

    model.eval()
    return model


EXP_NAMES = [
    'KSpace_patch',
    'KSpace_kaleidoscope',
    'KSpace_axial',
    'Lambda_Schedule_patch_cosine',
]
MODEL_LABELS = {
    'KSpace_patch':                 'KSpace Patch',
    'KSpace_kaleidoscope':          'KSpace Kaleidoscope',
    'KSpace_axial':                 'KSpace Axial',
    'Lambda_Schedule_patch_cosine': 'Baseline (Image Space)',
}

print('Loading models...')
models = {name: load_experiment(name) for name in EXP_NAMES}
print('Done.')

In [ ]:
# ── Inputs ─────────────────────────────────────────────────────────────────────
ph_np     = np.rot90(np.transpose(np.array(shepp_logan(N))), 1).astype(np.float32)
ph_tensor = torch.tensor(ph_np.copy()).unsqueeze(0).unsqueeze(0)  # [1,1,N,N]

dataset  = MRIDataset(DATA_DIR, N=N, split='val', val_fraction=0.1, seed=42)
real_img = dataset[0].unsqueeze(0)  # [1,1,N,N]

mask = load_mask(os.path.join(MASK_DIR, f'mask_R{ACCEL}.png'), N).to(device)


def simulate_undersampling(gt):
    gt    = gt.to(device)
    ks_f  = fft_2d(gt)
    ks_us = ks_f * mask
    zf    = ifft_2d(ks_us)[:, 0:1, :, :]
    return zf, ks_us


zf_phantom, ks_phantom = simulate_undersampling(ph_tensor)
zf_real,    ks_real    = simulate_undersampling(real_img)
gt_phantom = ph_tensor.to(device)
gt_real    = real_img.to(device)

In [ ]:
# ── Display helpers ────────────────────────────────────────────────────────────
def to_img(t):
    return np.abs(t[0, 0].cpu().numpy())


def to_ks(t):
    ks = fft_2d(t.to(device))
    r  = ks[0, 0].cpu().numpy()
    im = ks[0, 1].cpu().numpy()
    return np.log(np.fft.fftshift(np.sqrt(r**2 + im**2)) + 1e-8)


def mse_img(gt, recon):
    return (to_img(gt) - to_img(recon)) ** 2


def mse_ks(gt, recon):
    gt_ks    = fft_2d(gt.to(device))
    recon_ks = fft_2d(recon.to(device))
    diff_r   = (gt_ks[0, 0] - recon_ks[0, 0]).cpu().numpy()
    diff_i   = (gt_ks[0, 1] - recon_ks[0, 1]).cpu().numpy()
    return np.fft.fftshift(diff_r**2 + diff_i**2)


def calc_psnr(gt_im, recon_im, max_val=None):
    if max_val is None:
        max_val = gt_im.max()
    mse = np.mean((gt_im - recon_im) ** 2)
    return 20 * np.log10(max_val / (np.sqrt(mse) + 1e-12))


# ── Inference + plot ───────────────────────────────────────────────────────────
def run_and_plot(gt, zf, y, input_label):
    n_rows = len(EXP_NAMES) + 1          # +1 for the reference row at top
    fig, axes = plt.subplots(n_rows, 4, figsize=(16, 3.8 * n_rows))
    fig.suptitle(
        f'Inference Results — {input_label}  (R={ACCEL})',
        fontsize=13, fontweight='bold', y=1.01
    )

    gt_im    = to_img(gt)
    gt_ks    = to_ks(gt)
    zf_im    = to_img(zf)
    zf_ks    = to_ks(zf)
    max_val  = gt_im.max()
    vmax_img = np.percentile(gt_im, 99)
    vmin_ks, vmax_ks = np.percentile(gt_ks, 1), np.percentile(gt_ks, 99)

    # ── Row 0: reference ──────────────────────────────────────────────────────
    ref_data   = [gt_im,  gt_ks,  zf_im,  zf_ks]
    ref_cmaps  = ['gray', 'inferno', 'gray', 'inferno']
    ref_titles = ['GT Image', 'GT K-space', 'Undersampled Image', 'Undersampled K-space']
    ref_vlims  = [(0, vmax_img), (vmin_ks, vmax_ks), (0, vmax_img), (vmin_ks, vmax_ks)]

    for col, (data, cmap, title, (vmin, vmax)) in enumerate(
            zip(ref_data, ref_cmaps, ref_titles, ref_vlims)):
        axes[0, col].imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
        axes[0, col].set_title(title, fontsize=9, fontweight='bold')
        axes[0, col].axis('off')
    axes[0, 0].text(
        -0.04, 0.5, 'Reference', transform=axes[0, 0].transAxes,
        fontsize=9, fontweight='bold', va='center', ha='right', rotation=90,
    )

    # ── Column titles for model rows ──────────────────────────────────────────
    for col, t in enumerate(['Recon Image', 'Recon K-space', 'MSE — Image', 'MSE — K-space']):
        axes[1, col].set_title(t, fontsize=9, fontweight='bold')

    # ── Rows 1–N: one per model ───────────────────────────────────────────────
    print(f'\n--- PSNR  ({input_label}, R={ACCEL}) ---')
    for i, name in enumerate(EXP_NAMES):
        row = i + 1
        with torch.no_grad():
            recon = models[name](zf.to(device), y.to(device), mask).cpu()

        rc_im  = to_img(recon)
        rc_ks  = to_ks(recon)
        err_im = mse_img(gt, recon)
        err_ks = mse_ks(gt, recon)
        psnr   = calc_psnr(gt_im, rc_im, max_val)

        print(f'  {MODEL_LABELS[name]:30s}  PSNR = {psnr:.2f} dB')

        axes[row, 0].imshow(rc_im,  cmap='gray',    vmin=0,       vmax=vmax_img)
        axes[row, 1].imshow(rc_ks,  cmap='inferno', vmin=vmin_ks, vmax=vmax_ks)
        axes[row, 2].imshow(err_im, cmap='hot',     vmin=0,       vmax=np.percentile(err_im, 99))
        axes[row, 3].imshow(np.log(err_ks + 1e-8),  cmap='hot')

        for col in range(4):
            axes[row, col].axis('off')
        axes[row, 0].text(
            -0.04, 0.5, f"{MODEL_LABELS[name]}\nPSNR {psnr:.1f} dB",
            transform=axes[row, 0].transAxes,
            fontsize=8, fontweight='bold', va='center', ha='right', rotation=90,
        )

    plt.tight_layout()
    plt.show()


run_and_plot(gt_phantom, zf_phantom, ks_phantom, 'Shepp-Logan Phantom')
run_and_plot(gt_real,    zf_real,    ks_real,    'Real OASIS Image')

In [ ]:
import torch.nn.functional as F

p = 16 # 16
g = N // p       # 20  →  g×g = 400 patches

# Image domain: magnitude of the zero-filled real OASIS image
img_arr = to_img(zf_real)                                   # [N, N]

# K-space domain: log-magnitude, NO fftshift — honest view of what the model sees.
# fft_2d puts DC at the (0,0) corner, so corner patches = low-freq, centre patches = high-freq.
raw_r  = ks_real[0, 0].cpu().numpy()
raw_i  = ks_real[0, 1].cpu().numpy()
ks_raw = np.log(np.sqrt(raw_r**2 + raw_i**2) + 1e-8)       # [N, N], no fftshift


def make_mosaic(src, patch_size, border=1):
    g   = src.shape[0] // patch_size
    pb  = patch_size + border
    out = np.full((g * pb, g * pb), src.min(), dtype=np.float32)
    for i in range(g):
        for j in range(g):
            patch = src[i*patch_size:(i+1)*patch_size, j*patch_size:(j+1)*patch_size]
            out[i*pb:i*pb+patch_size, j*pb:j*pb+patch_size] = patch
    return out


mosaic_img = make_mosaic(img_arr, p)
mosaic_ks  = make_mosaic(ks_raw,  p)

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(
    f'Patch Tokenisation — Real OASIS Image  (R={ACCEL}, patch={p}×{p})\n'
    f'Row 0: what the Baseline (image-space) encoder receives  |  '
    f'Row 1: what the KSpace encoders receive (DC at corners, no fftshift)',
    fontsize=11, fontweight='bold'
)

rows = [
    # (source array, mosaic, cmap, row label, zoom region)
    (img_arr, mosaic_img, 'gray',    'Image domain\n(Baseline encoder input)',  'centre'),
    (ks_raw,  mosaic_ks,  'inferno', 'K-space domain\n(KSpace encoder input)',  'corner'),
]

for row_idx, (arr, mosaic, cmap, row_label, zoom_region) in enumerate(rows):
    vmin = arr.min()
    vmax = np.percentile(arr, 99)

    # col 0: full view + patch grid
    ax = axes[row_idx, 0]
    ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    for k in range(0, N + 1, p):
        ax.axhline(k - 0.5, color='cyan', linewidth=0.4, alpha=0.7)
        ax.axvline(k - 0.5, color='cyan', linewidth=0.4, alpha=0.7)
    ax.set_title(f'Full view + {p}×{p} grid', fontsize=9, fontweight='bold')
    ax.set_ylabel(row_label, fontsize=9, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

    # col 1: mosaic of all 400 patches
    ax = axes[row_idx, 1]
    ax.imshow(mosaic, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(f'All {g}×{g} = {g*g} patches\n(each cell = one token input)', fontsize=9, fontweight='bold')
    ax.axis('off')

    # col 2: zoom into representative 4×4 region
    zoom = 4
    if zoom_region == 'centre':
        c = g // 2
        lo, hi = c - zoom // 2, c + zoom // 2
        zoom_title = f'Centre {zoom}×{zoom} patches\n(brain tissue, mixed freq)'
    else:
        lo, hi = 0, zoom
        zoom_title = f'Corner {zoom}×{zoom} patches\n(DC / low-freq — bright because DC at corner)'

    ax = axes[row_idx, 2]
    ax.imshow(arr[lo*p:hi*p, lo*p:hi*p], cmap=cmap, vmin=vmin, vmax=vmax)
    for k in range(0, zoom * p + 1, p):
        ax.axhline(k - 0.5, color='cyan', linewidth=0.8, alpha=0.8)
        ax.axvline(k - 0.5, color='cyan', linewidth=0.8, alpha=0.8)
    ax.set_title(zoom_title, fontsize=9, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── Check if attention collapse is real or an observation artefact ─────────────
model    = models['KSpace_patch']
attn_mod = model.transformers[0].transformers[0].encoder.layers[0].self_attn

d  = attn_mod.embed_dim
W  = attn_mod.in_proj_weight   # [3*d, d]
b  = attn_mod.in_proj_bias

W_Q = W[:d]        # Q projection weights  [d, d]
W_K = W[d:2*d]     # K projection weights  [d, d]
W_V = W[2*d:]      # V projection weights  [d, d]

print('Projection weight norms:')
print(f'  W_Q  norm={W_Q.norm():.4f}  mean={W_Q.mean():.6f}  std={W_Q.std():.6f}')
print(f'  W_K  norm={W_K.norm():.4f}  mean={W_K.mean():.6f}  std={W_K.std():.6f}')
print(f'  W_V  norm={W_V.norm():.4f}  mean={W_V.mean():.6f}  std={W_V.std():.6f}')
print(f'  bias norm={b.norm():.4f}')
print()
print('If W_Q and W_K norms are near zero → weights genuinely collapsed during training.')
print('If W_V norm is normal but W_Q/W_K are zero → model learned to skip attention routing.')